In [5]:
# ============================================================
# ONE CELL: Train TrajTransformer + Save Artifacts + Launch UI (American Odds)
# Outputs files:
#   - epl_traj_artifacts.npz
#   - epl_traj_objects.pkl
#   - epl_traj_state.pt
# ============================================================

import os, math, random, glob, re, pickle, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import ipywidgets as widgets
from IPython.display import display, clear_output

warnings.filterwarnings("ignore")

# -------------------- CONFIG --------------------
CSV_PATH    = "epl_master_train_full.csv"
MIN_DATE    = "2005-01-01"
NAN_THRESH  = 0.90
RANDOM_SEED = 42

SEQ_LEN     = 12
BATCH_SIZE  = 256
EPOCHS      = 10          # adjust up if you want
LR          = 2e-4
WEIGHT_DECAY= 1e-2

D_MODEL     = 96
N_HEAD      = 4
N_LAYERS    = 3
D_FF        = 256
DROPOUT     = 0.2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CKPT_DIR = "checkpoints_epl_traj"
CKPT_PREFIX = "epl_traj"
os.makedirs(CKPT_DIR, exist_ok=True)
RESUME = True
SAVE_EVERY_N = 5  # save checkpoint every N epochs

print("DEVICE:", DEVICE)

# -------------------- helper: team name cleaning --------------------
def clean_team_name(x: str) -> str:
    if pd.isna(x):
        return x
    s = str(x).strip()
    repl = {
        "Wolverhampton": "Wolves",
        "Manchester City": "Man City",
        "Manchester United": "Man United",
        "West Ham United": "West Ham",
        "Nott'm Forest": "Nottm Forest",
        "Nottingham Forest": "Nottm Forest",
        "Spurs": "Tottenham",
    }
    return repl.get(s, s)

# -------------------- load + cleanup --------------------
if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Missing {CSV_PATH} in this runtime.")

df0 = pd.read_csv(CSV_PATH, low_memory=False)
df0["date"] = pd.to_datetime(df0.get("date", df0.get("Date", None)), errors="coerce")

rename_map = {
    "HomeTeam": "home_team", "AwayTeam": "away_team",
    "FTHG": "home_goals", "FTAG": "away_goals", "FTR": "ft_result"
}
for k, v in rename_map.items():
    if k in df0.columns and v not in df0.columns:
        df0 = df0.rename(columns={k: v})

df0 = df0.loc[:, df0.isna().mean() <= NAN_THRESH].copy()

need = ["date","home_team","away_team","home_goals","away_goals","target"]
missing = [c for c in need if c not in df0.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}\nFound: {list(df0.columns)[:30]}")

df0["home_team"] = df0["home_team"].map(clean_team_name)
df0["away_team"] = df0["away_team"].map(clean_team_name)

df0 = df0[df0["date"].notna() & (df0["date"] >= pd.to_datetime(MIN_DATE))].copy()
df0["target"] = df0["target"].astype(str).str.strip().str.upper()
df0 = df0[df0["target"].isin(["H","D","A"])].copy()

df0["home_goals"] = pd.to_numeric(df0["home_goals"], errors="coerce")
df0["away_goals"] = pd.to_numeric(df0["away_goals"], errors="coerce")
df0 = df0[df0["home_goals"].notna() & df0["away_goals"].notna()].copy()
df0 = df0.sort_values("date").reset_index(drop=True)

print("df0:", df0.shape, "date range:", df0["date"].min().date(), "->", df0["date"].max().date())

# -------------------- points helper --------------------
def result_points(hg, ag):
    if hg > ag:  return 3, 0
    if hg < ag:  return 0, 3
    return 1, 1

# ============================================================
# 1) Build long per-team history
# ============================================================
rows = []
for i, r in df0.iterrows():
    hg, ag = float(r["home_goals"]), float(r["away_goals"])
    hp, ap = result_points(hg, ag)
    dt = r["date"]

    rows.append({"match_idx": i, "date": dt, "team": r["home_team"], "opp": r["away_team"],
                 "is_home": 1, "gf": hg, "ga": ag, "gd": hg - ag, "pts": hp})
    rows.append({"match_idx": i, "date": dt, "team": r["away_team"], "opp": r["home_team"],
                 "is_home": 0, "gf": ag, "ga": hg, "gd": ag - hg, "pts": ap})

long = pd.DataFrame(rows).sort_values(["date","match_idx","team"]).reset_index(drop=True)

# ============================================================
# 2) Strictly causal table context
# ============================================================
teams = sorted(pd.unique(pd.concat([df0["home_team"], df0["away_team"]], ignore_index=True)))
team_to_id = {t:i for i,t in enumerate(teams)}

points = {t: 0.0 for t in teams}
gf_tot = {t: 0.0 for t in teams}
ga_tot = {t: 0.0 for t in teams}
last_date = {t: None for t in teams}

ctx_rows = []
for i, r in df0.iterrows():
    dt = r["date"]
    ht = r["home_team"]; at = r["away_team"]
    hg = float(r["home_goals"]); ag = float(r["away_goals"])
    hp, ap = result_points(hg, ag)

    table = []
    for t in teams:
        gd = gf_tot[t] - ga_tot[t]
        table.append((t, points[t], gd, gf_tot[t]))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    rank_map = {t: (rk+1) for rk, (t,_,_,_) in enumerate(table_sorted)}

    def rest_days(team):
        if last_date[team] is None:
            return np.nan
        return float((dt - last_date[team]).days)

    ctx_rows.append({
        "match_idx": i,
        "home_pts_pre": points[ht],
        "away_pts_pre": points[at],
        "home_rank_pre": rank_map[ht],
        "away_rank_pre": rank_map[at],
        "home_rest_days": rest_days(ht),
        "away_rest_days": rest_days(at),
    })

    points[ht] += hp; points[at] += ap
    gf_tot[ht] += hg; ga_tot[ht] += ag
    gf_tot[at] += ag; ga_tot[at] += hg
    last_date[ht] = dt; last_date[at] = dt

CTX_COLS = ["home_pts_pre","away_pts_pre","home_rank_pre","away_rank_pre","home_rest_days","away_rest_days"]

ctx = pd.DataFrame(ctx_rows).set_index("match_idx")
df = df0.join(ctx, how="left")

# ============================================================
# 3) Sequences
# ============================================================
long_by_team = {t: long[long["team"] == t].sort_values(["date","match_idx"]).reset_index(drop=True) for t in teams}
TOKEN_COLS = ["is_home", "gf", "ga", "gd", "pts"]

def get_team_seq(team: str, match_idx: int):
    hist = long_by_team[team]
    past = hist[hist["match_idx"] < match_idx]
    if len(past) < SEQ_LEN:
        return None
    return past.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)

def build_dataset_rows():
    out_rows = []
    for i, r in df.iterrows():
        ht, at = r["home_team"], r["away_team"]
        home_seq = get_team_seq(ht, i)
        away_seq = get_team_seq(at, i)
        if home_seq is None or away_seq is None:
            continue

        out_rows.append({
            "match_idx": i,
            "home_team": ht,
            "away_team": at,
            "home_seq": home_seq,
            "away_seq": away_seq,
            "home_pts_pre": float(r["home_pts_pre"]),
            "away_pts_pre": float(r["away_pts_pre"]),
            "home_rank_pre": float(r["home_rank_pre"]),
            "away_rank_pre": float(r["away_rank_pre"]),
            "home_rest_days": float(r["home_rest_days"]) if np.isfinite(r["home_rest_days"]) else np.nan,
            "away_rest_days": float(r["away_rest_days"]) if np.isfinite(r["away_rest_days"]) else np.nan,
            "target": r["target"],
        })
    return pd.DataFrame(out_rows)

data = build_dataset_rows().sort_values("match_idx").reset_index(drop=True)
for c in ["home_rest_days","away_rest_days"]:
    med = np.nanmedian(data[c].to_numpy())
    data[c] = data[c].fillna(med)

print("Matches with enough history:", len(data), "out of", len(df0))

# label encoding
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss
le = LabelEncoder()
y_enc = le.fit_transform(data["target"].values)
class_names = list(le.classes_)
print("Classes:", class_names)

# ============================================================
# 4) Time split
# ============================================================
split_idx = int(len(data) * 0.85)
train_df = data.iloc[:split_idx].copy()
test_df  = data.iloc[split_idx:].copy()

# ============================================================
# 5) Normalize using train stats
# ============================================================
token_stack = np.concatenate(train_df["home_seq"].values.tolist() + train_df["away_seq"].values.tolist(), axis=0)
tok_mean = token_stack.mean(axis=0)
tok_std  = token_stack.std(axis=0) + 1e-6

ctx_mat = train_df[CTX_COLS].to_numpy(dtype=np.float32)
ctx_mean = ctx_mat.mean(axis=0)
ctx_std  = ctx_mat.std(axis=0) + 1e-6

def norm_tokens(x): return (x - tok_mean) / tok_std
def norm_ctx(x):    return (x - ctx_mean) / ctx_std

# ============================================================
# 6) Dataset / Dataloader
# ============================================================
class EPLSeqDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, y: np.ndarray):
        self.f = frame.reset_index(drop=True)
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.f)

    def __getitem__(self, idx):
        r = self.f.iloc[idx]
        home_seq = norm_tokens(r["home_seq"]).astype(np.float32)
        away_seq = norm_tokens(r["away_seq"]).astype(np.float32)
        ctxv = norm_ctx(r[CTX_COLS].to_numpy(dtype=np.float32))
        home_id = team_to_id[r["home_team"]]
        away_id = team_to_id[r["away_team"]]
        return (
            torch.from_numpy(home_seq),
            torch.from_numpy(away_seq),
            torch.tensor(ctxv, dtype=torch.float32),
            torch.tensor(home_id, dtype=torch.long),
            torch.tensor(away_id, dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.long),
        )

y_train = le.transform(train_df["target"].values)
y_test  = le.transform(test_df["target"].values)

train_loader = DataLoader(EPLSeqDataset(train_df, y_train), batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
test_loader  = DataLoader(EPLSeqDataset(test_df,  y_test),  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# ============================================================
# 7) Model
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        T = x.size(1)
        return x + self.pe[:, :T, :]

class TrajTransformer(nn.Module):
    def __init__(self, token_dim, n_teams, ctx_dim, n_classes):
        super().__init__()
        self.team_emb = nn.Embedding(n_teams, D_MODEL)
        self.in_proj = nn.Linear(token_dim, D_MODEL)
        self.pos = PositionalEncoding(D_MODEL, max_len=SEQ_LEN)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=D_FF,
            dropout=DROPOUT, batch_first=True, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=N_LAYERS)

        self.cls = nn.Parameter(torch.zeros(1, 1, D_MODEL))
        nn.init.normal_(self.cls, std=0.02)

        self.ctx_mlp = nn.Sequential(
            nn.Linear(ctx_dim, D_MODEL),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(D_MODEL, D_MODEL),
        )

        self.head = nn.Sequential(
            nn.Linear(D_MODEL*3, 256),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(256, n_classes)
        )

    def encode_team(self, seq, team_id):
        B, T, _ = seq.shape
        x = self.in_proj(seq)
        x = self.pos(x)
        cls = self.cls.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)
        te = self.team_emb(team_id).unsqueeze(1)
        x = x + te
        z = self.encoder(x)
        return z[:, 0, :]

    def forward(self, home_seq, away_seq, ctxv, home_id, away_id):
        h = self.encode_team(home_seq, home_id)
        a = self.encode_team(away_seq, away_id)
        c = self.ctx_mlp(ctxv)
        feat = torch.cat([h, a, c], dim=1)
        return self.head(feat)

token_dim = len(TOKEN_COLS)
ctx_dim = len(CTX_COLS)
n_classes = len(class_names)

model = TrajTransformer(token_dim, n_teams=len(teams), ctx_dim=ctx_dim, n_classes=n_classes).to(DEVICE)

# ============================================================
# 8) Train (RESUMABLE)
# ============================================================
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

criterion = nn.CrossEntropyLoss()
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    all_logits, all_y = [], []
    for home_seq, away_seq, ctxv, home_id, away_id, y in loader:
        home_seq = home_seq.to(DEVICE)
        away_seq = away_seq.to(DEVICE)
        ctxv = ctxv.to(DEVICE)
        home_id = home_id.to(DEVICE)
        away_id = away_id.to(DEVICE)
        y = y.to(DEVICE)

        logits = model(home_seq, away_seq, ctxv, home_id, away_id)
        loss = criterion(logits, y)

        if train:
            optim.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

        total_loss += float(loss.item()) * y.size(0)
        all_logits.append(logits.detach().cpu())
        all_y.append(y.detach().cpu())

    all_logits = torch.cat(all_logits, dim=0).numpy()
    all_y = torch.cat(all_y, dim=0).numpy()
    probs = torch.softmax(torch.from_numpy(all_logits), dim=1).numpy()
    return total_loss / len(loader.dataset), log_loss(all_y, probs)

def ckpt_path(epoch: int) -> str:
    return os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_epoch_{epoch:03d}.pt")

def find_latest_checkpoint():
    files = glob.glob(os.path.join(CKPT_DIR, f"{CKPT_PREFIX}_epoch_*.pt"))
    if not files:
        return None
    def epnum(p):
        m = re.search(r"_epoch_(\d+)\.pt$", p)
        return int(m.group(1)) if m else -1
    files = sorted(files, key=epnum)
    return files[-1]

def save_checkpoint(epoch, best_test, best_state):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optim_state_dict": optim.state_dict(),
        "best_test": float(best_test),
        "best_state": best_state,
    }
    path = ckpt_path(epoch)
    torch.save(payload, path)
    print(f"💾 Saved checkpoint: {path}")

def load_checkpoint(path: str):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optim.load_state_dict(ckpt["optim_state_dict"])
    for state in optim.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(DEVICE)
    start_epoch = int(ckpt["epoch"]) + 1
    best_test = float(ckpt.get("best_test", 1e9))
    best_state = ckpt.get("best_state", None)
    print(f"✅ Resumed from: {path} (start_epoch={start_epoch}, best_test={best_test:.6f})")
    return start_epoch, best_test, best_state

start_epoch = 1
best_test = 1e9
best_state = None

if RESUME:
    latest = find_latest_checkpoint()
    if latest is not None:
        start_epoch, best_test, best_state = load_checkpoint(latest)
    else:
        print("ℹ️ No checkpoint found. Training from scratch.")

if best_state is None:
    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

for ep in range(start_epoch, start_epoch + EPOCHS):
    tr_loss, tr_ll = run_epoch(train_loader, train=True)
    te_loss, te_ll = run_epoch(test_loader, train=False)
    print(f"Epoch {ep:02d} | train loss {tr_loss:.4f} ll {tr_ll:.4f} | test loss {te_loss:.4f} ll {te_ll:.4f}")

    if te_ll < best_test:
        best_test = te_ll
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"⭐ New best test log loss: {best_test:.6f}")

    if (ep % SAVE_EVERY_N) == 0:
        save_checkpoint(ep, best_test, best_state)

model.load_state_dict(best_state)
print("✅ Best test log loss:", best_test)

# ============================================================
# 9) Temperature scaling calibration
# ============================================================
class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.logT = nn.Parameter(torch.zeros(()))
    def forward(self, logits):
        T = torch.exp(self.logT) + 1e-6
        return logits / T

scaler = TemperatureScaler().to(DEVICE)
optT = torch.optim.LBFGS(scaler.parameters(), lr=0.1, max_iter=50)

model.eval()
test_logits = []
test_y = []
with torch.no_grad():
    for home_seq, away_seq, ctxv, home_id, away_id, y in test_loader:
        logits = model(home_seq.to(DEVICE), away_seq.to(DEVICE), ctxv.to(DEVICE), home_id.to(DEVICE), away_id.to(DEVICE))
        test_logits.append(logits)
        test_y.append(y.to(DEVICE))

test_logits = torch.cat(test_logits, dim=0)
test_y = torch.cat(test_y, dim=0)

def closure():
    optT.zero_grad()
    loss = nn.CrossEntropyLoss()(scaler(test_logits), test_y)
    loss.backward()
    return loss

optT.step(closure)
Tcal = float(torch.exp(scaler.logT).detach().cpu())
print("✅ Calibrated Temperature T:", Tcal)

# ============================================================
# 10) Save artifacts for UI (the 3 files you were missing)
# ============================================================
# Objects needed by UI
obj = {
    "df0": df0,
    "clean_team_name": clean_team_name,
    "long_by_team": long_by_team,
    "TOKEN_COLS": TOKEN_COLS,
    "SEQ_LEN": SEQ_LEN,
    "team_to_id": team_to_id,
    "class_names": class_names,
    "CTX_COLS": CTX_COLS,
}
with open("epl_traj_objects.pkl", "wb") as f:
    pickle.dump(obj, f)

np.savez(
    "epl_traj_artifacts.npz",
    tok_mean=np.asarray(tok_mean),
    tok_std=np.asarray(tok_std),
    ctx_mean=np.asarray(ctx_mean),
    ctx_std=np.asarray(ctx_std),
    CTX_COLS=np.asarray(list(CTX_COLS), dtype=object),
    TOKEN_COLS=np.asarray(list(TOKEN_COLS), dtype=object),
    SEQ_LEN=np.asarray(int(SEQ_LEN)),
    class_names=np.asarray(list(class_names), dtype=object),
)

torch.save(
    {"model_state": model.state_dict(), "scaler_state": scaler.state_dict()},
    "epl_traj_state.pt"
)

print("✅ Saved: epl_traj_objects.pkl, epl_traj_artifacts.npz, epl_traj_state.pt")

# ============================================================
# 11) UI: enter American odds -> BET / NO BET
# ============================================================
# --- small utils ---
def _is_finite(x): return x is not None and np.isfinite(x)

def _entropy(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1.0)
    return float(-(p * np.log(p)).sum())

def _confidence_margin(p_dict, labs=("H","D","A")):
    vals = np.array([float(p_dict.get(k, np.nan)) for k in labs], dtype=float)
    vals = np.where(np.isfinite(vals), vals, 0.0)
    s = np.sort(vals)[::-1]
    return float(s[0] - s[1]) if len(s) >= 2 else 0.0

def _american_to_decimal(a: float) -> float:
    return 1.0 + (a / 100.0) if a > 0 else 1.0 + (100.0 / abs(a))

def _parse_odds(s: str, mode: str):
    s = (s or "").strip()
    if s == "":
        return None
    s2 = s.replace(" ", "")
    try:
        if mode == "Decimal":
            x = float(s2)
            return float(x) if x >= 1.01 else None
        if mode == "American":
            x = float(s2)
            if x == 0:
                return None
            # if user typed 110 (no sign), interpret as +110
            if not s2.startswith("+") and not s2.startswith("-") and x > 0:
                x = +x
            return float(_american_to_decimal(x))
        # Auto
        if s2.startswith("+") or s2.startswith("-"):
            return float(_american_to_decimal(float(s2)))
        x = float(s2)
        if 1.01 <= x <= 25.0:
            return float(x)
        return float(_american_to_decimal(x))
    except Exception:
        return None

def implied_probs_from_decimal_odds(dec_h, dec_d, dec_a):
    if not (_is_finite(dec_h) and _is_finite(dec_d) and _is_finite(dec_a)):
        return None, None, None, None, None
    p_raw = np.array([1/dec_h, 1/dec_d, 1/dec_a], dtype=float)
    s = float(p_raw.sum())
    overround = float(s - 1.0)
    p_fair = p_raw / (s + 1e-12)
    return float(p_fair[0]), float(p_fair[1]), float(p_fair[2]), float(overround), float(s)

# --- normalization for UI inference ---
ctx_mean_np = np.asarray(ctx_mean, dtype=np.float32).copy()
ctx_std_np  = np.asarray(ctx_std, dtype=np.float32).copy()
def norm_tokens_np(x: np.ndarray) -> np.ndarray:
    return (x - tok_mean) / tok_std
def norm_ctx_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return (x - ctx_mean_np) / ctx_std_np

# strictly-causal table builder AS-OF date (no leakage)
def build_table_state_asof(df0_in: pd.DataFrame, asof_dt: pd.Timestamp):
    dfx = df0_in.copy()
    dfx["date"] = pd.to_datetime(dfx["date"], errors="coerce")
    dfx = dfx[dfx["date"].notna()].copy()
    dfx["home_team"] = dfx["home_team"].map(clean_team_name)
    dfx["away_team"] = dfx["away_team"].map(clean_team_name)
    dfx = dfx[dfx["date"] < pd.to_datetime(asof_dt)].sort_values("date").reset_index(drop=True)

    teams_local = sorted(pd.unique(pd.concat([df0_in["home_team"], df0_in["away_team"]], ignore_index=True)).tolist())
    teams_local = [clean_team_name(t) for t in teams_local]
    teams_local = sorted(list(dict.fromkeys(teams_local)))

    points_l = {t: 0.0 for t in teams_local}
    gf_l = {t: 0.0 for t in teams_local}
    ga_l = {t: 0.0 for t in teams_local}
    last_l = {t: None for t in teams_local}

    for _, r in dfx.iterrows():
        dt = r["date"]
        ht = clean_team_name(r["home_team"])
        at = clean_team_name(r["away_team"])
        hg = float(r["home_goals"]); ag = float(r["away_goals"])
        hp, ap = result_points(hg, ag)

        points_l[ht] += hp; points_l[at] += ap
        gf_l[ht] += hg; ga_l[ht] += ag
        gf_l[at] += ag; ga_l[at] += hg
        last_l[ht] = dt; last_l[at] = dt

    table = []
    for t in teams_local:
        gd = gf_l[t] - ga_l[t]
        table.append((t, points_l[t], gd, gf_l[t]))
    table_sorted = sorted(table, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    rank = {t: (rk + 1) for rk, (t, _, _, _) in enumerate(table_sorted)}

    return points_l, gf_l, ga_l, last_l, rank

def fixture_ctx_asof_full(asof_dt, ht, at, df0_in):
    asof_dt = pd.to_datetime(asof_dt)
    ht = clean_team_name(ht); at = clean_team_name(at)

    points_l, gf_l, ga_l, last_l, rank_l = build_table_state_asof(df0_in, asof_dt)

    def rest(team):
        ld = last_l.get(team, None)
        if ld is None:
            return np.nan
        return float((asof_dt - pd.to_datetime(ld)).days)

    known = {
        "home_pts_pre": float(points_l.get(ht, 0.0)),
        "away_pts_pre": float(points_l.get(at, 0.0)),
        "home_rank_pre": float(rank_l.get(ht, len(team_to_id))),
        "away_rank_pre": float(rank_l.get(at, len(team_to_id))),
        "home_rest_days": rest(ht),
        "away_rest_days": rest(at),
    }

    # start from mean; overwrite keys we have
    c = ctx_mean_np.copy()
    col_to_idx = {col: i for i, col in enumerate(CTX_COLS)}
    for k, v in known.items():
        if k in col_to_idx and np.isfinite(v):
            c[col_to_idx[k]] = float(v)

    # clip rests if present
    for k in ["home_rest_days", "away_rest_days"]:
        if k in col_to_idx:
            idx = col_to_idx[k]
            if not np.isfinite(c[idx]):
                c[idx] = ctx_mean_np[idx]
            c[idx] = float(np.clip(c[idx], 0.0, 30.0))

    return c.astype(np.float32)

def latest_team_seq(team: str):
    team = clean_team_name(team)
    hist = long_by_team.get(team, None)
    if hist is None or len(hist) < SEQ_LEN:
        return None
    return hist.iloc[-SEQ_LEN:][TOKEN_COLS].to_numpy(dtype=np.float32)

def predict_fixture_transformer(dt, ht, at):
    hs = latest_team_seq(ht)
    asq = latest_team_seq(at)
    if hs is None or asq is None:
        return None
    c = fixture_ctx_asof_full(dt, ht, at, df0)

    hs_t = torch.from_numpy(norm_tokens_np(hs)).unsqueeze(0).to(DEVICE)
    as_t = torch.from_numpy(norm_tokens_np(asq)).unsqueeze(0).to(DEVICE)
    c_t  = torch.from_numpy(norm_ctx_np(c)).unsqueeze(0).to(DEVICE)

    hid = torch.tensor([team_to_id[clean_team_name(ht)]], dtype=torch.long).to(DEVICE)
    aid = torch.tensor([team_to_id[clean_team_name(at)]], dtype=torch.long).to(DEVICE)

    model.eval(); scaler.eval()
    with torch.no_grad():
        logits = model(hs_t, as_t, c_t, hid, aid)
        logits = scaler(logits)
        p = torch.softmax(logits, dim=1).cpu().numpy()[0]
    return dict(zip(class_names, p.tolist()))

def _kelly_fraction(p, dec):
    b = float(dec) - 1.0
    if b <= 0:
        return 0.0
    f = (float(dec) * float(p) - 1.0) / b
    return float(np.clip(f, 0.0, 1.0))

# -------------------- UI widgets --------------------
teams_ui = sorted(pd.unique(pd.concat([df0["home_team"], df0["away_team"]], ignore_index=True)).tolist())
teams_ui = [clean_team_name(t) for t in teams_ui]
teams_ui = sorted(list(dict.fromkeys(teams_ui)))

date_picker = widgets.DatePicker(description="Date:", value=pd.Timestamp.today().date())
home_dd = widgets.Dropdown(options=teams_ui, description="Home:",
                           value=(teams_ui[0] if len(teams_ui) else None),
                           layout=widgets.Layout(width="320px"))
away_dd = widgets.Dropdown(options=teams_ui, description="Away:",
                           value=(teams_ui[1] if len(teams_ui) > 1 else teams_ui[0]),
                           layout=widgets.Layout(width="320px"))

odds_mode = widgets.Dropdown(options=["Auto","American","Decimal"], value="American",
                             description="Odds format:", layout=widgets.Layout(width="260px"))

odds_home = widgets.Text(description="Odds H:", value="", placeholder="+110", layout=widgets.Layout(width="260px"))
odds_draw = widgets.Text(description="Odds D:", value="", placeholder="+240", layout=widgets.Layout(width="260px"))
odds_away = widgets.Text(description="Odds A:", value="", placeholder="-120", layout=widgets.Layout(width="260px"))

allow_draw = widgets.Checkbox(description="Allow Draw bets", value=True)

min_conf_margin = widgets.FloatSlider(description="Min conf (p1-p2)", min=0.0, max=0.40, step=0.01, value=0.05,
                                      readout_format=".2f", layout=widgets.Layout(width="420px"))
min_edge = widgets.FloatSlider(description="Min edge", min=0.0, max=0.15, step=0.005, value=0.02,
                               readout_format=".3f", layout=widgets.Layout(width="420px"))
min_ev = widgets.FloatSlider(description="Min EV", min=-0.10, max=0.20, step=0.005, value=0.01,
                             readout_format=".3f", layout=widgets.Layout(width="420px"))
max_overround = widgets.FloatSlider(description="Max vig (overround)", min=0.00, max=0.20, step=0.005, value=0.08,
                                    readout_format=".3f", layout=widgets.Layout(width="420px"))

use_kelly = widgets.Checkbox(description="Use fractional Kelly sizing", value=True)
kelly_frac = widgets.FloatSlider(description="Kelly fraction", min=0.0, max=1.0, step=0.05, value=0.25,
                                 readout_format=".2f", layout=widgets.Layout(width="420px"))
bankroll = widgets.FloatText(description="Bankroll ($):", value=100.0, layout=widgets.Layout(width="260px"))
flat_stake = widgets.FloatText(description="Flat stake ($):", value=2.0, layout=widgets.Layout(width="260px"))

btn = widgets.Button(description="Predict", button_style="primary", icon="check")
out = widgets.Output()

def _banner(ok: bool, lines: list, stake_line: str=""):
    icon = "✅" if ok else "❌"
    print(f"\n{icon} {'BET' if ok else 'NO BET'}")
    for ln in lines:
        print(" -", ln)
    if stake_line:
        print(" -", stake_line)

def on_click(_):
    with out:
        clear_output(wait=True)
        dt = date_picker.value
        ht = home_dd.value
        at = away_dd.value

        if dt is None:
            print("⚠️ Pick a date.")
            return
        if ht == at:
            print("⚠️ Home and Away can’t be the same team.")
            return

        p = predict_fixture_transformer(dt, ht, at)
        if p is None:
            print("⚠️ Not enough team history for SEQ_LEN =", SEQ_LEN)
            return

        probs = pd.Series({k: float(p.get(k, 0.0)) for k in ["H","D","A"]})
        probs = probs / (probs.sum() + 1e-12)

        ent = _entropy(probs.values)
        margin = _confidence_margin(probs.to_dict(), labs=("H","D","A"))

        print(f"{pd.to_datetime(dt).date()} | {clean_team_name(ht)} vs {clean_team_name(at)}")
        print(f"Model probs: H={probs['H']:.3f}  D={probs['D']:.3f}  A={probs['A']:.3f}")
        print(f"Confidence: margin={margin:.3f} | entropy={ent:.3f}")

        reasons = []
        if margin < float(min_conf_margin.value):
            reasons.append(f"Confidence margin {margin:.3f} < {min_conf_margin.value:.3f}")

        mode = odds_mode.value
        dec_h = _parse_odds(odds_home.value, mode)
        dec_d = _parse_odds(odds_draw.value, mode)
        dec_a = _parse_odds(odds_away.value, mode)

        have_all_odds = _is_finite(dec_h) and _is_finite(dec_d) and _is_finite(dec_a)
        if not have_all_odds:
            _banner(False, reasons + ["Enter valid H/D/A odds to compute EV/edge (realistic mode)."])
            return

        qH, qD, qA, ov, raw_sum = implied_probs_from_decimal_odds(dec_h, dec_d, dec_a)
        print(f"\nOdds (decimal): H={dec_h:.3f} D={dec_d:.3f} A={dec_a:.3f}")
        print(f"Market fair probs: qH={qH:.3f} qD={qD:.3f} qA={qA:.3f}")
        print(f"Vig overround={ov:.3f} (raw sum={raw_sum:.3f})")

        if ov > float(max_overround.value):
            reasons.append(f"Vig {ov:.3f} > {max_overround.value:.3f}")

        rows = []
        for lab, name, dec, q in [("H","Home (H)",dec_h,qH), ("D","Draw (D)",dec_d,qD), ("A","Away (A)",dec_a,qA)]:
            if lab == "D" and not allow_draw.value:
                continue
            p_model = float(probs[lab])
            edge = p_model - float(q)
            ev = (p_model * float(dec)) - 1.0
            rows.append([lab, name, p_model, float(q), edge, ev, float(dec)])

        table = pd.DataFrame(rows, columns=["lab","Outcome","p_model","p_market_fair","edge","EV","dec_odds"])
        best = table.loc[table["EV"].idxmax()].copy()

        if float(best["EV"]) < float(min_ev.value):
            reasons.append(f"Best EV {best['EV']:.3f} < {min_ev.value:.3f}")
        if float(best["edge"]) < float(min_edge.value):
            reasons.append(f"Best edge {best['edge']:.3f} < {min_edge.value:.3f}")

        ok = (len(reasons) == 0)
        stake_line = ""
        if ok:
            BR = float(bankroll.value or 0.0)
            if use_kelly.value and BR > 0:
                fstar = _kelly_fraction(best["p_model"], best["dec_odds"])
                stake_amt = BR * float(kelly_frac.value) * fstar
                stake_line = f"Stake ${stake_amt:.2f} (Kelly f*={fstar:.3f}, fraction={kelly_frac.value:.2f})"
            else:
                stake_amt = float(flat_stake.value or 0.0)
                stake_line = f"Stake ${stake_amt:.2f} (flat)"
            _banner(True, [f"Best side: {best['Outcome']}",
                           f"EV={best['EV']:.3f} | edge={best['edge']:.3f} | odds={best['dec_odds']:.3f}"], stake_line)
        else:
            _banner(False, reasons + [f"Best side was {best['Outcome']} (EV={best['EV']:.3f}, edge={best['edge']:.3f})"])

        display(table.style.format({
            "p_model":"{:.3f}",
            "p_market_fair":"{:.3f}",
            "edge":"{:.3f}",
            "EV":"{:.3f}",
            "dec_odds":"{:.3f}"
        }))

btn.on_click(on_click)

ui_row1 = widgets.HBox([date_picker, home_dd, away_dd])
ui_row2 = widgets.HBox([odds_mode, odds_home, odds_draw, odds_away])
ui_row3 = widgets.HBox([allow_draw, bankroll, flat_stake])
ui_row4 = widgets.HBox([min_conf_margin, min_edge])
ui_row5 = widgets.HBox([min_ev, max_overround])
ui_row6 = widgets.HBox([use_kelly, kelly_frac, btn])

display(ui_row1, ui_row2, ui_row3, ui_row4, ui_row5, ui_row6, out)

print("\n✅ All-in-one done: trained model + calibrated scaler + saved artifacts + UI ready.")

DEVICE: cpu
df0: (7871, 190) date range: 2005-08-13 -> 2026-02-23
Matches with enough history: 7463 out of 7871
Classes: ['A', 'D', 'H']
ℹ️ No checkpoint found. Training from scratch.
Epoch 01 | train loss 1.0237 ll 1.0237 | test loss 1.0246 ll 1.0246
⭐ New best test log loss: 1.024588
Epoch 02 | train loss 0.9786 ll 0.9786 | test loss 1.0178 ll 1.0178
⭐ New best test log loss: 1.017761
Epoch 03 | train loss 0.9681 ll 0.9681 | test loss 1.0191 ll 1.0191
Epoch 04 | train loss 0.9633 ll 0.9633 | test loss 1.0189 ll 1.0189
Epoch 05 | train loss 0.9630 ll 0.9630 | test loss 1.0271 ll 1.0271
💾 Saved checkpoint: checkpoints_epl_traj/epl_traj_epoch_005.pt
Epoch 06 | train loss 0.9593 ll 0.9593 | test loss 1.0217 ll 1.0217
Epoch 07 | train loss 0.9558 ll 0.9558 | test loss 1.0204 ll 1.0204
Epoch 08 | train loss 0.9514 ll 0.9514 | test loss 1.0190 ll 1.0190
Epoch 09 | train loss 0.9504 ll 0.9504 | test loss 1.0172 ll 1.0172
⭐ New best test log loss: 1.017176
Epoch 10 | train loss 0.9504 ll 0.95

Output()


✅ All-in-one done: trained model + calibrated scaler + saved artifacts + UI ready.
